# Kunjungan Negara

Ammar, ketua Persatuan AI Indonesia, gemar sekali bepergian ke luar negeri untuk menjalin relasi dan memperkenalkan potensi AI Indonesia kepada dunia. Setelah berkeliling ke berbagai negara, kali ini ia bersiap mengunjungi Belanda.

Untuk mempersiapkan kunjungannya, Ammar mengumpulkan berbagai teks berbahasa Indonesia beserta terjemahannya dalam bahasa Belanda. Ia ingin mempelajari pasangan teks tersebut agar lebih mudah memahami bahan bacaan dan berkomunikasi dengan calon mitranya di sana.

Sayangnya, saat berkas-berkas itu digabungkan, urutan teks berbahasa Belanda justru teracak! Semua teks dan terjemahannya masih lengkap, tetapi catatan yang menghubungkan setiap pasangan hilang.

Bantulah Ammar mencocokkan kembali setiap teks berbahasa Indonesia dengan terjemahannya dalam bahasa Belanda agar ia dapat melanjutkan persiapan kunjungannya!


## Tugas

Setiap testcase memberikan `N` teks bahasa Indonesia dan `N` teks bahasa Belanda. Setiap teks Indonesia memiliki satu pasangan terjemahannya dalam dataset berbahasa Belanda, tetapi urutan teks Belanda telah diacak.

Implementasikan fungsi:

```python
def pasangkan(
    indonesian: list[str],
    dutch: list[str],
) -> list[int]:
    ...
```

Nilai `answer[i]` harus berisi indeks teks Belanda yang merupakan terjemahan dari `indonesian[i]`.

Contoh:

```python
indonesian = ["Saya membaca buku.", "Dia membeli roti."]
dutch = ["Hij koopt brood.", "Ik lees een boek."]

answer = [1, 0]
```

Jawaban harus merupakan permutasi lengkap indeks `0` sampai `N-1`. Dipastikan tiap teks memiliki pasangan.


## Data dan penilaian

- Data latih berisi 10.000 pasangan teks Indonesia dan Belanda.
- Data uji berisi 150 testcase tanpa kunci jawaban.
- Setiap testcase menggunakan `N = 20`.
- Data mencampurkan pasangan sintetis dan pasangan dari corpus nyata.
- Tidak ada teks evaluasi yang muncul persis di data latih.

Skor sebuah testcase adalah proporsi posisi matching yang benar. Skor akhir
adalah rata-rata skor seluruh testcase. Jawaban yang bukan permutasi lengkap
indeks 0 sampai 19 mendapat skor nol untuk testcase tersebut.


In [2]:
# Setup
from pathlib import Path
import hashlib
import random

import numpy as np
import pandas as pd

SEED = 2026
N_SENTENCES = 20


def stable_seed(*parts):
    payload = "\x1f".join(map(str, parts)).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "big")


def cari_folder_data(berkas_wajib):
    """Cari folder dataset pada komputer lokal atau Kaggle."""
    kandidat = [Path("kaggle_data"), Path("../kaggle_data"), Path(".")]
    folder_kaggle = Path("/kaggle/input")
    if folder_kaggle.exists():
        kandidat.extend(path.parent for path in folder_kaggle.rglob(berkas_wajib))
    for folder in kandidat:
        if (folder / berkas_wajib).exists() and (folder / "sample_submission.csv").exists():
            return folder
    raise FileNotFoundError(f"Dataset Kaggle dengan berkas {berkas_wajib} tidak ditemukan")


JALUR_SUBMISSION = (
    Path("/kaggle/working/submission.csv")
    if Path("/kaggle/working").exists()
    else Path("submission.csv")
)

DATA_DIR = cari_folder_data("train.csv")
print("Folder data:", DATA_DIR.resolve())


Folder data: /kaggle/input/competitions/soal-2-3-final-ekka-2026/kaggle_data


## Memuat dataset Kaggle

`train.csv` menyediakan pasangan terjemahan untuk pembelajaran. Pada `test.csv`,
setiap testcase disimpan sebagai 40 baris: 20 teks Indonesia dan 20 teks Belanda.
Kolom `sentence_index` mempertahankan indeks asli di dalam masing-masing bahasa.


In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

TRAIN_PAIRS = train.to_dict("records")
EVAL_CASES = []
for case_id in sample_submission["id"]:
    rows = test.loc[test["case_id"] == case_id]
    indonesian_rows = rows.loc[rows["language"] == "indonesian"].sort_values("sentence_index")
    dutch_rows = rows.loc[rows["language"] == "dutch"].sort_values("sentence_index")
    if (
        indonesian_rows["sentence_index"].tolist() != list(range(N_SENTENCES))
        or dutch_rows["sentence_index"].tolist() != list(range(N_SENTENCES))
    ):
        raise ValueError(f"Indeks kalimat tidak lengkap untuk testcase {case_id}")
    EVAL_CASES.append({
        "id": case_id,
        "indonesian": indonesian_rows["text"].tolist(),
        "dutch": dutch_rows["text"].tolist(),
    })

print("Train pairs:", len(TRAIN_PAIRS))
print("Test cases :", len(EVAL_CASES))


Train pairs: 10000
Test cases : 250


In [4]:
train

,id,indonesian,dutch
0,train_pair_00001,Arsitek itu menggantikan perawat itu di rumah ...,Vanmorgen vervangt de architect de verpleegkun...
1,train_pair_00002,Tur konser adalah suatu rangkaian konser oleh ...,Een concerttournee is een reeks voorstellingen...
2,train_pair_00003,Film ini merupakan sekuel dari film tahun 2013...,De film is het vervolg op Pacific Rim uit 2013.
3,train_pair_00004,PaaS persembahan juga dapat mencakup fasilitas...,Het PaaS-aanbod kan ook faciliteiten omvatten ...
4,train_pair_00005,Alarm pengaman atau alarm keamanan adalah alar...,Een alarmsysteem of alarminstallatie is een el...
...,...,...,...
9995,train_pair_09996,Aliran material yang sangat panas membuat air ...,Zeer heet materiaal van de pyroclastische wolk...
9996,train_pair_09997,Ruang kelas itu bising minggu lalu sebab ada p...,Het klaslokaal was vorige week lawaaierig omda...
9997,train_pair_09998,Frekuensi kedatangan kereta adalah sekitar 7 m...,Treinen rijden met een frequentie van ongeveer...
9998,train_pair_09999,Teknisi yang menelepon ilmuwan itu sedang menu...,"De technicus die de wetenschapper belt, wacht ..."


In [20]:
allindodoc = []
alldutchdoc = []
allindovocab = []
alldutchvocab = []
punct = {"?": "", "!": "", ".": "", ",":"", "-": "", "'":"", "#":"", "`":""}
for _, r in train.iterrows():
    indo = r[["indonesian"]].tolist()[0]
    dutch = r[["dutch"]].tolist()[0]
    indo = indo.lower()
    dutch = dutch.lower()
    for long, new in punct.items():
        indo = indo.replace(long, new)
        dutch = dutch.replace(long, new)
    allindodoc.append(indo)
    alldutchdoc.append(dutch)
    for word in indo.split():
        allindovocab.append(word)
    for word in dutch.split():
        alldutchvocab.append(word)
allindovocab = list(set(allindovocab))
alldutchvocab = list(set(alldutchvocab))

In [21]:
for case in EVAL_CASES:
    indlist = case["indonesian"]
    for doc in indlist:
        indo = doc.lower()
        for long, new in punct.items():
            indo = indo.replace(long, new)
        allindodoc.append(indo)
        for word in indo.split():
            allindovocab.append(word)
    dutchlist = case["dutch"]
    for doc in dutchlist:
        dutch = doc.lower()
        for long, new in punct.items():
            dutch = dutch.lower().replace(long, new)
        alldutchdoc.append(dutch)
        for word in dutch.split():
            alldutchvocab.append(word)

In [22]:

allindovocab = list(set(allindovocab))
alldutchvocab = list(set(alldutchvocab))

In [ ]:
tf = []
for i in range(10000, 15000):
    katadidoc = allindodoc[i].split()
    jumlahkatadidoc = len(katadidoc)
    tfperdoc = []
    for word in allindovocab:
        cnt = 0
        for docword in katadidoc:
            if docword == word:
                cnt+=1
        tfperdoc.append(cnt)
    tf.append(tfperdoc)
    print(i)

In [48]:
idf = []
for word in allindovocab:
    cnt = 0
    for doc in allindodoc:
        if word in doc:
            cnt+=1
    idword = np.log(len(allindodoc)/(cnt+1))+1
    idf.append(idword)

In [49]:
tf = np.asarray(tf)
idf = np.asarray(idf)

In [50]:
tfidf = tf*idf

In [28]:
tfd = []
for i in range(10000, 15000):
    katadidoc = alldutchdoc[i].split()
    jumlahkatadidoc = len(katadidoc)
    tfperdoc = []
    for word in alldutchvocab:
        cnt = 0
        for docword in katadidoc:
            if docword == word:
                cnt+=1
        tfperdoc.append(cnt)
    tfd.append(tfperdoc)

In [47]:
idfd = []
for word in alldutchvocab:
    cnt = 0
    for doc in alldutchdoc:
        if word in doc:
            cnt+=1
    idword = np.log(len(alldutchdoc)/(cnt+1))+1
    idfd.append(idword)
tfd = np.asarray(tfd)
idfd = np.asarray(idfd)

In [51]:
tfd = np.asarray(tfd)

In [52]:
tfidfd = tfd*idfd

In [53]:
from collections import defaultdict
allpasangan = []
for idx in range(0,5000,20):
    indovect = []
    for idxind in range(20):
        indovect.append(tfidf[idx+idxind])
    dutchvect = []
    for idxdutch in range(20):
        dutchvect.append(tfidfd[idx+idxdutch])
    bigsimilarity = []
    indovect = np.asarray(indovect)
    dutchvect = np.asarray(dutchvect)
    for i in range(20):
        ls = []
        for j in range(20):
            similarity = np.sum(indovect[i] * dutchvect[j, :19004])/ (np.linalg.norm(indovect[i]) * np.linalg.norm(dutchvect[j, :19004]))
            ls.append(similarity)
        bigsimilarity.append(ls)
    bigsimilarity = np.asarray(bigsimilarity)
    score = bigsimilarity.copy()
    pasangan = defaultdict(int)
    match = 0
    while match <20:
        arged = np.argmax(score)
        i = arged //len(score)
        j = arged %len(score)
        pasangan[j] = i
        score[i, :] = -1
        score[:, j] = -1
        match+=1
    ans = ""
    for i in range(20):
        val = pasangan[i]
        ans = f"{ans} {val}"
    allpasangan.append(ans[1:])

In [129]:
baseans = ""
for i in range(20):
    baseans = f"{baseans} 0"
baseans = baseans[1:]

In [132]:
# for i in range(20):
allpasangan = [baseans] * 250
filnum = 45
answers = []
for case in EVAL_CASES:
    answer = pasangkan_acak(case["indonesian"], case["dutch"])
    if not valid_matching(answer):
        raise ValueError(f"Matching tidak valid untuk testcase {case['id']}")
    answers.append(" ".join(map(str, answer)))
FILENAME = f"submission{filnum}.csv"
JALUR_SUBMISSION = (
    Path(f"/kaggle/working/{FILENAME}")
    if Path("/kaggle/working").exists()
    else Path(f"{FILENAME}")
)
submission = sample_submission.copy()
submission["matching"] = allpasangan
if submission.isna().any().any():
    raise ValueError("Submission mengandung nilai kosong")
submission.to_csv(JALUR_SUBMISSION, index=False)

print("Submission ditulis ke:", JALUR_SUBMISSION)
display(submission.head())

Submission ditulis ke: /kaggle/working/submission45.csv


,id,matching
0,case_0001,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
1,case_0002,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
2,case_0003,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
3,case_0004,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
4,case_0005,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0


In [54]:
answers = []
# for case in EVAL_CASES:
#     answer = pasangkan_acak(case["indonesian"], case["dutch"])
#     if not valid_matching(answer):
#         raise ValueError(f"Matching tidak valid untuk testcase {case['id']}")
#     answers.append(" ".join(map(str, answer)))
FILENAME = f"submission55.csv"
JALUR_SUBMISSION = (
    Path(f"/kaggle/working/{FILENAME}")
    if Path("/kaggle/working").exists()
    else Path(f"{FILENAME}")
)
submission = sample_submission.copy()
submission["matching"] = allpasangan
if submission.isna().any().any():
    raise ValueError("Submission mengandung nilai kosong")
submission.to_csv("submission101.csv", index=False)

print("Submission ditulis ke:", JALUR_SUBMISSION)
display(submission.head())


Submission ditulis ke: /kaggle/working/submission55.csv


,id,matching
0,case_0001,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
1,case_0002,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
2,case_0003,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
3,case_0004,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
4,case_0005,11 0 7 1 2 3 4 5 6 8 9 10 12 13 14 15 16 17 18 19


In [131]:
submission

,id,matching
0,case_0001,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
1,case_0002,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
2,case_0003,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
3,case_0004,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
4,case_0005,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
...,...,...
245,case_0246,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
246,case_0247,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
247,case_0248,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
248,case_0249,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0


In [121]:
sample_submission

,id,matching
0,case_0001,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
1,case_0002,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
2,case_0003,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
3,case_0004,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
4,case_0005,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
...,...,...
245,case_0246,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
246,case_0247,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
247,case_0248,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
248,case_0249,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


In [16]:
print("Contoh training pair:")
display(train[["indonesian", "dutch"]].head())

example = EVAL_CASES[0]
print("\nContoh testcase:", example["id"])
display(pd.DataFrame({
    "indonesian_index": range(N_SENTENCES),
    "indonesian": example["indonesian"],
}).head())
display(pd.DataFrame({
    "dutch_index": range(N_SENTENCES),
    "dutch": example["dutch"],
}).head())


Contoh training pair:


,indonesian,dutch
0,Arsitek itu menggantikan perawat itu di rumah ...,Vanmorgen vervangt de architect de verpleegkun...
1,Tur konser adalah suatu rangkaian konser oleh ...,Een concerttournee is een reeks voorstellingen...
2,Film ini merupakan sekuel dari film tahun 2013...,De film is het vervolg op Pacific Rim uit 2013.
3,PaaS persembahan juga dapat mencakup fasilitas...,Het PaaS-aanbod kan ook faciliteiten omvatten ...
4,Alarm pengaman atau alarm keamanan adalah alar...,Een alarmsysteem of alarminstallatie is een el...



Contoh testcase: case_0001


,indonesian_index,indonesian
0,0,Tas hitam itu lebih mahal daripada tas merah itu.
1,1,Mereka mengirim surat penting itu di rumah sor...
2,2,Jalur sepeda itu tidak sepanjang jalan utama i...
3,3,Novel itu sama tebalnya dengan buku merah itu ...
4,4,Tolong jangan gambar potret itu di taman sebel...


,dutch_index,dutch
0,0,Leg na de lunch de nieuwe regel alstublieft ni...
1,1,De technicus legt vandaag de procedure op scho...
2,2,De zwarte tas is duurder dan de rode tas.
3,3,De bureaulamp wordt tijdens de pauze door de m...
4,4,De belangrijke brief wordt op school door de j...


In [10]:
def valid_matching(answer, size=N_SENTENCES):
    if not isinstance(answer, (list, tuple, np.ndarray)) or len(answer) != size:
        return False
    if not all(isinstance(item, (int, np.integer)) and not isinstance(item, bool) for item in answer):
        return False
    return sorted(map(int, answer)) == list(range(size))


## Baseline acak

Baseline mengembalikan permutasi acak yang deterministik. Nilai harapannya sekitar `0.05`.


In [11]:
def pasangkan_acak(indonesian, dutch):
    rng = random.Random(stable_seed(*indonesian, *dutch))
    answer = list(range(len(dutch)))
    rng.shuffle(answer)
    return answer


## Membuat `submission.csv`

Kolom `matching` berisi 20 indeks yang dipisahkan spasi. Elemen ke-i adalah
indeks teks Belanda yang dipasangkan dengan teks Indonesia ke-i.


In [12]:
answers = []
for case in EVAL_CASES:
    answer = pasangkan_acak(case["indonesian"], case["dutch"])
    if not valid_matching(answer):
        raise ValueError(f"Matching tidak valid untuk testcase {case['id']}")
    answers.append(" ".join(map(str, answer)))

submission = sample_submission.copy()
submission["matching"] = answers
if submission.isna().any().any():
    raise ValueError("Submission mengandung nilai kosong")
submission.to_csv(JALUR_SUBMISSION, index=False)

print("Submission ditulis ke:", JALUR_SUBMISSION)
display(submission.head())


Submission ditulis ke: /kaggle/working/submission.csv


,id,matching
0,case_0001,18 6 9 19 7 13 17 3 1 2 10 12 11 14 8 5 0 4 16 15
1,case_0002,4 17 6 8 18 15 5 12 19 1 14 3 16 10 0 2 11 7 13 9
2,case_0003,6 16 17 10 9 7 11 2 12 0 14 19 4 1 18 13 15 3 8 5
3,case_0004,3 9 0 1 12 6 4 18 15 7 8 10 11 2 14 13 5 19 17 16
4,case_0005,6 8 17 0 18 16 19 2 10 12 13 1 5 15 9 3 7 11 4 14


In [27]:
newtext = "abc?. ilo"
newtext.replace("?", "")

'abc. ilo'

In [17]:
example = EVAL_CASES[0]
print("\nContoh testcase:", example["id"])
display(pd.DataFrame({
    "indonesian_index": range(N_SENTENCES),
    "indonesian": example["indonesian"],
}).head())
display(pd.DataFrame({
    "dutch_index": range(N_SENTENCES),
    "dutch": example["dutch"],
}).head())



Contoh testcase: case_0001


,indonesian_index,indonesian
0,0,Tas hitam itu lebih mahal daripada tas merah itu.
1,1,Mereka mengirim surat penting itu di rumah sor...
2,2,Jalur sepeda itu tidak sepanjang jalan utama i...
3,3,Novel itu sama tebalnya dengan buku merah itu ...
4,4,Tolong jangan gambar potret itu di taman sebel...


,dutch_index,dutch
0,0,Leg na de lunch de nieuwe regel alstublieft ni...
1,1,De technicus legt vandaag de procedure op scho...
2,2,De zwarte tas is duurder dan de rode tas.
3,3,De bureaulamp wordt tijdens de pauze door de m...
4,4,De belangrijke brief wordt op school door de j...
